# Vast.ai — YouTube Transcripts em 4 GPUs

Notebook operacional para preparar o ambiente, validar 4 GPUs e executar um smoke test do runner `scripts/vastai_transcribe_4gpu.py`.

**A rodada completa deve ser executada no Terminal do Jupyter dentro de `tmux`**, seguindo `docs/VASTAI_4GPU_TRANSCRIPTS.md`.

## 1. Confirmar as GPUs
Esperado: quatro GPUs visíveis.

In [ ]:
!nvidia-smi -L

import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## 2. Instalar o projeto e bibliotecas CUDA
Execute esta célula somente depois de abrir o notebook dentro do repositório clonado.

In [ ]:
%pip install -e ".[dev]"
%pip install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

In [ ]:
import os
import nvidia.cublas.lib
import nvidia.cudnn.lib

cuda_libs = (
    os.path.dirname(nvidia.cublas.lib.__file__)
    + ':'
    + os.path.dirname(nvidia.cudnn.lib.__file__)
)
os.environ['LD_LIBRARY_PATH'] = cuda_libs + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print(os.environ['LD_LIBRARY_PATH'])

## 3. Testes do repositório
Não avance se houver falhas.

In [ ]:
!pytest -q

## 4. Preparar o arquivo de entrada
Faça upload do `videos.parquet` local para:

`input/lideres_opiniao_credito_2025_2026/videos.parquet`

In [ ]:
from pathlib import Path
import pandas as pd

INPUT = Path('input/lideres_opiniao_credito_2025_2026/videos.parquet')
OUTPUT = Path('outputs/lideres_opiniao_credito_2025_2026')
OUTPUT.mkdir(parents=True, exist_ok=True)

assert INPUT.exists(), f'Arquivo não encontrado: {INPUT}'
videos = pd.read_parquet(INPUT)
print('linhas:', len(videos))
print('vídeos únicos:', videos['video_id'].nunique())
display(videos[['video_id', 'channel_title']].head())

## 5. Pré-baixar o modelo
Isso evita quatro downloads simultâneos quando os workers iniciarem.

In [ ]:
from faster_whisper import download_model
model_path = download_model('small')
print(model_path)

## 6. Smoke test com 8 vídeos
O runner usa fila compartilhada, quatro processos e checkpoint por GPU.

In [ ]:
import subprocess

cmd = [
    'python', 'scripts/vastai_transcribe_4gpu.py',
    '--config', 'configs/lideres_opiniao_credito_2025_2026.yaml',
    '--videos', str(INPUT),
    '--output-dir', str(OUTPUT),
    '--gpus', '0,1,2,3',
    '--model', 'small',
    '--compute-type', 'float16',
    '--limit', '8',
]
subprocess.run(cmd, check=True, env=os.environ.copy())

## 7. Validar o smoke test

In [ ]:
import json

summary = json.loads((OUTPUT / 'summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
transcripts = pd.read_parquet(OUTPUT / 'transcripts.parquet')
print(transcripts['transcript_status'].value_counts(dropna=False))
print('\nMétodos:')
print(transcripts['transcript_method'].value_counts(dropna=False))
display(transcripts[['video_id', 'transcript_status', 'transcript_method', 'transcript_language']].head(20))

## 8. Rodada completa

Se o smoke test estiver correto, **não rode a carga longa nesta célula**. Abra um Terminal do Jupyter e siga a seção **Rodada completa em tmux** de `docs/VASTAI_4GPU_TRANSCRIPTS.md`.

O comando completo é o mesmo, mas sem `--limit 8`. O runner retomará automaticamente os sucessos do smoke test.